# Disaster Response AI Platform — Full-Dataset GPU Training Pipeline
### End-to-End Model Training, Calibration, Quality Gating & Registry Checkpointing on Google Colab (T4/A100)

This notebook trains the Disaster Response AI multi-stage computer vision models on the **complete, full-scale datasets** using GPU acceleration (`--device cuda`):

1. **Stage 1 Edge Triage**: MobileNetV3-Small on all 6,433+ real aerial UAV images from **AIDER** (Kyrkou et al.).
2. **Stage 2 Structural Damage**: 4-Tier Ordinal Damage CNN on full building crops from **RescueNet** (Bina-Lab Hurricane Ian UAV).
3. **Stage 2 Road Passability**: Binary Accessibility Classifier on full RGB road scenes from **RescueNet**.
4. **Stage 2 Flood Extent U-Net**: 2D Convolutional U-Net on all paired high-resolution UAV images & masks from **FloodNet-Supervised v1.0** (Rahnemoonfar et al.).

**Provenance & Versioning Protocol**: Full-dataset training automatically promotes models to **v2.0.0** in `config/model_registry.json` and updates `models/benchmark_report.json` with single-source-of-truth canonical metrics.

## 1. Hardware & Environment Verification
Verifies NVIDIA GPU availability and CUDA acceleration. Ensure your Colab runtime is set to **GPU** (`Runtime > Change runtime type > T4 GPU`).

In [1]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "[HARDWARE ERROR] No GPU detected! Please navigate to: "
        "Runtime > Change runtime type > T4 GPU (or A100) before proceeding."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"[OK] Active Compute Device: {gpu_name} ({vram_gb:.1f} GB VRAM)")

Wed Sep 16 08:09:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Repository & Install Dependencies
Clones the latest code from GitHub and installs necessary computer vision and evaluation packages.

In [2]:
# Clone or pull latest repository
!git clone https://github.com/YMP7/Disaster-Response.git || (cd Disaster-Response && git pull)
%cd Disaster-Response

# Install dependencies
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q opencv-python-headless numpy pytest pytest-asyncio anyio

print("[OK] Dependencies installed successfully.")

Cloning into 'Disaster-Response'...
remote: Enumerating objects: 177, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 177 (delta 93), reused 120 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (177/177), 150.46 KiB | 1.22 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/Disaster-Response
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.4 MB/s eta 0:00:00
[OK] Dependencies installed successfully.


## 3. Automated Dataset Download & Fail-Loud Verification

This cell downloads and unpacks the 3 disaster benchmark datasets:
- **AIDER**: Via Kaggle API (`clguo1/aiderdata`). Requires `kaggle.json`.
- **FloodNet**: Via Dropbox Direct Archive (`?dl=1`).
- **RescueNet**: Via Kaggle API (`ymonishprasanna/rescuenet`), or fallback Dropbox link.

> **Fail-Loud Safety Guard**: If any dataset is missing, truncated, or below expected file count thresholds, this cell **aborts immediately** with clear instructions rather than allowing training to run on incomplete data.


In [3]:
import os
import sys
from pathlib import Path
import subprocess

data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# A. AIDER Dataset Download (~2.5 GB)
# ---------------------------------------------------------------------
aider_dir = data_dir / "AIDER"
aider_files = list(aider_dir.rglob("*.jpg")) if aider_dir.exists() else []

if len(aider_files) < 5000:
    print("\n=== [1/3] Downloading AIDER Dataset from Kaggle ===")

    # Check for ~/.kaggle/access_token, ~/.kaggle/kaggle.json, or environment variable
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_cfg = kaggle_dir / "kaggle.json"
    kaggle_token = kaggle_dir / "access_token"

    # If access_token file exists, read into KAGGLE_API_TOKEN
    if kaggle_token.exists():
        token_val = kaggle_token.read_text(encoding="utf-8").strip()
        if token_val:
            os.environ["KAGGLE_API_TOKEN"] = token_val
            print(f"[OK] Loaded Kaggle API Access Token from {kaggle_token}")

    has_auth = (
        kaggle_cfg.exists()
        or kaggle_token.exists()
        or bool(os.environ.get("KAGGLE_API_TOKEN"))
        or (bool(os.environ.get("KAGGLE_USERNAME")) and bool(os.environ.get("KAGGLE_KEY")))
    )

    if not has_auth:
        try:
            from google.colab import files
            print("Kaggle credentials not detected.")
            print("You can paste your Kaggle Access Token (starts with KGAT_) or upload kaggle.json:")
            user_input = input("Enter Kaggle Token (or press Enter to upload file): ").strip()
            if user_input:
                kaggle_dir.mkdir(parents=True, exist_ok=True)
                kaggle_token.write_text(user_input, encoding="utf-8")
                os.chmod(kaggle_token, 0o600)
                os.environ["KAGGLE_API_TOKEN"] = user_input
                print("[OK] Kaggle Access Token saved and configured.")
            else:
                uploaded = files.upload()
                if "kaggle.json" in uploaded:
                    kaggle_dir.mkdir(parents=True, exist_ok=True)
                    with open(kaggle_cfg, "wb") as f:
                        f.write(uploaded["kaggle.json"])
                    os.chmod(kaggle_cfg, 0o600)
                    print("[OK] kaggle.json configured.")
                elif "access_token" in uploaded:
                    kaggle_dir.mkdir(parents=True, exist_ok=True)
                    with open(kaggle_token, "wb") as f:
                        f.write(uploaded["access_token"])
                    os.chmod(kaggle_token, 0o600)
                    token_val = kaggle_token.read_text(encoding="utf-8").strip()
                    os.environ["KAGGLE_API_TOKEN"] = token_val
                    print("[OK] access_token configured.")
        except Exception as e:
            print(f"Note: {e}")

    # Install/upgrade kaggle library to latest version
    !pip install -q --upgrade kaggle

    # Download dataset
    !kaggle datasets download -d clguo1/aiderdata -p /tmp/aider
    !mkdir -p data/AIDER && unzip -q /tmp/aider/*.zip -d data/AIDER/
    !rm -rf /tmp/aider
else:
    print(f"[OK] AIDER dataset already present ({len(aider_files)} images).")


# ---------------------------------------------------------------------
# B. FloodNet Dataset Download (~5.2 GB)
# ---------------------------------------------------------------------
flood_dir = data_dir / "FloodNet"
flood_files = list(flood_dir.rglob("*.jpg")) if flood_dir.exists() else []

if len(flood_files) < 1500:
    print("\n=== [2/3] Downloading FloodNet-Supervised v1.0 from Dropbox Archive ===")
    flood_url = "https://www.dropbox.com/scl/fo/k33qdif15ns2qv2jdxvhx/ANGaa8iPRhvlrvcKXjnmNRc?rlkey=ao2493wzl1cltonowjdbrnp7f&e=5&dl=1"
    !mkdir -p /tmp/floodnet
    !curl -L -o /tmp/floodnet/floodnet.zip "$flood_url"

    # Verify downloaded file is a valid zip (not an HTML error page)
    if not os.path.exists("/tmp/floodnet/floodnet.zip") or os.path.getsize("/tmp/floodnet/floodnet.zip") < 1000000:
        raise RuntimeError(
            "\n" + "!" * 80 + "\n"
            "[DATASET ERROR] FloodNet Dropbox link returned an invalid or rate-limited archive.\n"
            "Please download FloodNet manually from:\n"
            "https://www.dropbox.com/scl/fo/k33qdif15ns2qv2jdxvhx/ANGaa8iPRhvlrvcKXjnmNRc?rlkey=ao2493wzl1cltonowjdbrnp7f&e=5&dl=0\n"
            "and upload/unzip into 'data/FloodNet/'.\n"
            + "!" * 80
        )

    !mkdir -p data/FloodNet && unzip -q /tmp/floodnet/floodnet.zip -d data/FloodNet/
    !rm -rf /tmp/floodnet
else:
    print(f"[OK] FloodNet dataset already present ({len(flood_files)} images).")


# ---------------------------------------------------------------------
# C. RescueNet Dataset Download (~10.4 GB)
# ---------------------------------------------------------------------
rescue_dir = data_dir / "RescueNet"
rescue_files = list(rescue_dir.rglob("*.png")) if rescue_dir.exists() else []

if len(rescue_files) < 2000:
    print("\n=== [3/3] Downloading RescueNet Post-Hurricane Ian UAV Dataset from Kaggle ===")
    !mkdir -p /tmp/rescuenet

    # Primary: Download from Kaggle dataset ymonishprasanna/rescuenet
    download_success = False
    try:
        res = subprocess.run(
            ["kaggle", "datasets", "download", "-d", "ymonishprasanna/rescuenet", "-p", "/tmp/rescuenet"],
            capture_output=True, text=True
        )
        print(res.stdout)
        if res.returncode == 0 and any(Path("/tmp/rescuenet").glob("*.zip")):
            download_success = True
            print("[OK] Downloaded RescueNet from Kaggle.")
        else:
            print(f"[NOTE] Kaggle download message: {res.stderr.strip() or res.stdout.strip()}")
    except Exception as e:
        print(f"[NOTE] Kaggle CLI call failed: {e}")

    # Fallback to Dropbox direct archive if Kaggle download wasn't successful
    if not download_success:
        print("[INFO] Attempting RescueNet download via Dropbox fallback...")
        rescue_url = "https://www.dropbox.com/scl/fo/ntgeyhxe2mzd2wuh7he7x/AHJ-cNzQL-Eu04HS6bvBgcw?rlkey=6vxiaqve9gp6vzvzh3t5mz0vv&e=6&dl=1"
        !curl -L -o /tmp/rescuenet/rescuenet.zip "$rescue_url"
        if os.path.exists("/tmp/rescuenet/rescuenet.zip") and os.path.getsize("/tmp/rescuenet/rescuenet.zip") >= 1000000:
            download_success = True

    if not download_success or not any(Path("/tmp/rescuenet").glob("*.zip")):
        raise RuntimeError(
            "\n" + "!" * 80 + "\n"
            "[DATASET ERROR] RescueNet download failed from both Kaggle and Dropbox.\n"
            "Please download RescueNet manually from Kaggle:\n"
            "  https://www.kaggle.com/datasets/ymonishprasanna/rescuenet/data\n"
            "or Dropbox:\n"
            "  https://www.dropbox.com/scl/fo/ntgeyhxe2mzd2wuh7he7x/AHJ-cNzQL-Eu04HS6bvBgcw?rlkey=6vxiaqve9gp6vzvzh3t5mz0vv&e=6&dl=0\n"
            "and upload/unzip into 'data/RescueNet/'.\n"
            + "!" * 80
        )

    !mkdir -p data/RescueNet && unzip -q /tmp/rescuenet/*.zip -d data/RescueNet/
    !rm -rf /tmp/rescuenet
else:
    print(f"[OK] RescueNet dataset already present ({len(rescue_files)} masks).")


# ---------------------------------------------------------------------
# D. Strict Dataset Audit & Threshold Verification
# ---------------------------------------------------------------------
final_aider = len(list(aider_dir.rglob("*.*")))
final_flood = len(list(flood_dir.rglob("*.*")))
final_rescue = len(list(rescue_dir.rglob("*.*")))

print("\n" + "=" * 60)
print("DATASET VERIFICATION AUDIT REPORT")
print("=" * 60)
print(f"AIDER Files:     {final_aider} (Expected >= 5,000)")
print(f"FloodNet Files:  {final_flood} (Expected >= 5,000)")
print(f"RescueNet Files: {final_rescue} (Expected >= 10,000)")

assert final_aider >= 5000, f"AIDER file count ({final_aider}) is below expected full dataset threshold!"
assert final_flood >= 5000, f"FloodNet file count ({final_flood}) is below expected full dataset threshold!"
assert final_rescue >= 10000, f"RescueNet file count ({final_rescue}) is below expected full dataset threshold!"
print("\n[PASSED] All datasets verified complete and ready for full-scale GPU training.")



=== [1/3] Downloading AIDER Dataset from Kaggle ===
Kaggle credentials not detected.
You can paste your Kaggle Access Token (starts with KGAT_) or upload kaggle.json:
Enter Kaggle Token (or press Enter to upload file): KGAT_946b06574a1e9f38e9af0b22dad02661
[OK] Kaggle Access Token saved and configured.
Dataset URL: https://www.kaggle.com/datasets/clguo1/aiderdata
License(s): unknown
100% 263M/263M [00:02<00:00, 103MB/s]


=== [2/3] Downloading FloodNet-Supervised v1.0 from Dropbox Archive ===
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0      3      0  0:00:05  0:00:05 --:--:--     4
100 12.1G  100 12.1G    0     0  21.5M      0  0:09:38  0:09:38 --:--:-- 5034k
mapname:  conversion of  failed

=== [3/3] Downloading RescueNet Post-Hurricane Ian UAV Dataset from Kaggle ===
Dataset URL: https://www.kaggle.com/datasets/ymonishprasanna/rescuenet
Li

## 3.5 Per-Class File Count Sanity Check

Prints a detailed breakdown of **every subdirectory** in each dataset root so you can visually confirm the exact class distribution before committing to the multi-hour training run.

- Any class with **0 files** is flagged with ⚠️.
- Review the tables below *before* running Cell 4. If any folder looks wrong (empty class, unexpected nesting, missing split), investigate before training.

In [5]:
from pathlib import Path
from collections import defaultdict

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def count_image_files(directory: Path) -> dict:
    """Count image files per immediate subdirectory (one level deep).
    Also counts root-level image files if any exist."""
    counts = {}
    if not directory.exists():
        return {'[MISSING — directory does not exist]': 0}
    for child in sorted(directory.iterdir()):
        if child.is_dir():
            n = sum(1 for f in child.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
            counts[child.name] = n
        elif child.is_file() and child.suffix.lower() in IMAGE_EXTS:
            counts['(root-level files)'] = counts.get('(root-level files)', 0) + 1
    if not counts:
        counts['[EMPTY — no subdirectories or image files found]'] = 0
    return counts


def print_dataset_table(title: str, counts: dict):
    """Print a formatted per-class file count table with warning flags."""
    col_w = max(len(k) for k in counts) + 2
    total = sum(counts.values())
    print(f'\n{"═" * 64}')
    print(f'  {title}')
    print(f'{"═" * 64}')
    print(f'  {"Subfolder / Class":<{col_w}}  {"Files":>8}')
    print(f'  {"─" * col_w}  {"─" * 8}')
    for name, n in counts.items():
        flag = '  ⚠️ EMPTY' if n == 0 else ''
        print(f'  {name:<{col_w}}  {n:>8}{flag}')
    print(f'  {"─" * col_w}  {"─" * 8}')
    print(f'  {"TOTAL":<{col_w}}  {total:>8}')


data_dir = Path('data')

# ── AIDER ──────────────────────────────────────────────────────────
# AIDER typically has class folders: collapsed_building, fire, flood,
# normal, traffic_incident (each with train/test subfolders).
aider_root = data_dir / 'AIDER'
aider_counts = count_image_files(aider_root)
# If AIDER has nested train/test splits per class, expand one more level
if aider_root.exists():
    expanded = {}
    for child in sorted(aider_root.iterdir()):
        if child.is_dir():
            sub_dirs = [d for d in child.iterdir() if d.is_dir()]
            if sub_dirs:  # Has sub-splits (train/test)
                for sub in sorted(sub_dirs):
                    n = sum(1 for f in sub.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
                    expanded[f'{child.name}/{sub.name}'] = n
            else:  # Flat class folder
                n = sum(1 for f in child.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
                expanded[child.name] = n
    if expanded:
        aider_counts = expanded
print_dataset_table('AIDER — Aerial Image Dataset for Emergency Response', aider_counts)


# ── FloodNet ───────────────────────────────────────────────────────
# FloodNet-Supervised typically has train/val/test splits,
# each containing 'image' and 'mask' subdirectories.
flood_root = data_dir / 'FloodNet'
flood_counts = {}
if flood_root.exists():
    for split_dir in sorted(flood_root.iterdir()):
        if split_dir.is_dir():
            sub_dirs = [d for d in split_dir.iterdir() if d.is_dir()]
            if sub_dirs:
                for sub in sorted(sub_dirs):
                    n = sum(1 for f in sub.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
                    flood_counts[f'{split_dir.name}/{sub.name}'] = n
            else:
                n = sum(1 for f in split_dir.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
                flood_counts[split_dir.name] = n
    root_imgs = sum(1 for f in flood_root.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
    if root_imgs:
        flood_counts['(root-level files)'] = root_imgs
else:
    flood_counts['[MISSING — directory does not exist]'] = 0
print_dataset_table('FloodNet-Supervised v1.0 — UAV Flood Images + Masks', flood_counts)


# ── RescueNet ──────────────────────────────────────────────────────
# RescueNet typically has train/val/test splits,
# each containing 'image' and 'label' (mask) subdirectories.
rescue_root = data_dir / 'RescueNet'
rescue_counts = {}
if rescue_root.exists():
    for split_dir in sorted(rescue_root.iterdir()):
        if split_dir.is_dir():
            sub_dirs = [d for d in split_dir.iterdir() if d.is_dir()]
            if sub_dirs:
                for sub in sorted(sub_dirs):
                    n = sum(1 for f in sub.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
                    rescue_counts[f'{split_dir.name}/{sub.name}'] = n
            else:
                n = sum(1 for f in split_dir.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
                rescue_counts[split_dir.name] = n
    root_imgs = sum(1 for f in rescue_root.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
    if root_imgs:
        rescue_counts['(root-level files)'] = root_imgs
else:
    rescue_counts['[MISSING — directory does not exist]'] = 0
print_dataset_table('RescueNet — Post-Hurricane Ian UAV Dataset', rescue_counts)


# ── Summary Gate ──────────────────────────────────────────────────
print(f'\n{"═" * 64}')
print('  ✓  REVIEW THE TABLES ABOVE before running Cell 4.')
print('  ✓  Any class with 0 files or a ⚠️ flag needs investigation.')
print('  ✓  Confirm image/mask counts are paired (same count per split).')
print(f'{"═" * 64}')


════════════════════════════════════════════════════════════════
  AIDER — Aerial Image Dataset for Emergency Response
════════════════════════════════════════════════════════════════
  Subfolder / Class              Files
  ──────────────────────────  ────────
  AIDER/collapsed_building         511
  AIDER/fire                       521
  AIDER/flooded_areas              526
  AIDER/normal                    4390
  AIDER/traffic_incident           485
  ──────────────────────────  ────────
  TOTAL                           6433

════════════════════════════════════════════════════════════════
  FloodNet-Supervised v1.0 — UAV Flood Images + Masks
════════════════════════════════════════════════════════════════
  Subfolder / Class                                 Files
  ─────────────────────────────────────────────  ────────
  ColorMasks-FloodNetv1.0/ColorMasks-TestSet          448
  ColorMasks-FloodNetv1.0/ColorMasks-TrainSet        1445
  ColorMasks-FloodNetv1.0/ColorMasks-ValSet    

## 4. Full-Dataset GPU Training Execution

Executes `models/train.py` with the new flags:
- `--full-dataset`: Removes all sample caps (loads all 6,433 AIDER, all RescueNet crops & scenes, and all FloodNet masks).
- `--device cuda`: Runs all model forward & backward passes on the GPU.
- `--batch-size 32`: Standardized batch size for GPU parallelization.
- Quality gates: Road passability classifier must outperform chance level (> 60%) to be marked active.

In [6]:
!python models/train.py \
    --full-dataset \
    --device cuda \
    --batch-size 32 \
    --epochs-s1 3 \
    --epochs-s2 8 \
    --epochs-flood 5

[Trainer] Device: cuda | Full Dataset: True | Batch Size: 32
STARTING END-TO-END TRAINING & VALIDATION PIPELINE (device=cuda, full_dataset=True)

--- [TRAIN STAGE 1] AIDER Aerial Scene Triage ---
Loading real AIDER dataset from: data/AIDER (full=True)
Dataset split: 5144 train samples, 1289 val samples
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth
100% 9.83M/9.83M [00:00<00:00, 112MB/s]
  Epoch 1/3 - Loss: 0.4618 (23.8s)
  Epoch 2/3 - Loss: 0.2478 (18.4s)
  Epoch 3/3 - Loss: 0.2119 (19.9s)
[OK] Saved Stage 1 weights: models/weights/stage1_mobilenetv3_india_v2.pt

--- [TRAIN STAGE 2] RescueNet Structural Damage & Road Accessibility ---
Extracting building crops from RescueNet: data/RescueNet (full=True)
Dataset crops: 8704 train crops, 1004 val crops
  [Damage Head] Epoch 1/8 - Loss: 1.0534 (1.7s)
  [Damage Head] Epoch 2/8 - Loss: 0.8935 (1.4s)
  [Damage Head] Epoch 3/8 - Loss: 0.8

## 5. Test Suite & Cross-File Canonical Consistency Verification

Runs the entire 27-test automated test suite directly on the newly trained v2.0.0 checkpoints to verify:
- Zero regression across all modules.
- Checkpoint isolation and clean initialization.
- Canonical consistency across `benchmark_report.json` and `model_registry.json`.
- Road passability quality gate enforcement.

In [7]:
!pytest -v tests/

import json
with open("config/model_registry.json") as f:
    registry = json.load(f)

print("\nActive Models in Registry:")
for stage, model_id in registry.get("active_models", {}).items():
    meta = registry["models"][model_id]
    print(f"  - {stage}: {model_id} (version: {meta['version']}, status: {meta['status']})")

with open("models/benchmark_report.json") as f:
    report = json.load(f)
print(f"\nFull Training Elapsed Time: {report.get('elapsed_seconds', 0.0):.1f} seconds")

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/Disaster-Response
configfile: pytest.ini
plugins: asyncio-1.4.0, anyio-4.14.2, langsmith-0.12.1, typeguard-4.6.0
asyncio: mode=Mode.AUTO, debug=False, asyncio_default_fixture_loop_scope=function, asyncio_default_test_loop_scope=function
collected 27 items                                                             

tests/test_end_to_end.py::TestDataPipelineAndPrivacy::test_bounding_box_validation PASSED [  3%]
tests/test_end_to_end.py::TestDataPipelineAndPrivacy::test_india_domain_augmentation PASSED [  7%]
tests/test_end_to_end.py::TestDataPipelineAndPrivacy::test_dpdp_privacy_filter PASSED [ 11%]
tests/test_end_to_end.py::TestDataPipelineAndPrivacy::test_synthetic_generator PASSED [ 14%]
tests/test_end_to_end.py::TestComputerVisionAndFusion::test_stage1_edge_triage PASSED [ 18%]
tes

## 6. Packaging & Exporting Trained Artifacts

Bundles the trained `.pt` model weights, `model_registry.json`, and `benchmark_report.json` into a deployable zip archive for seamless download and integration back into your local repository or production service.

In [8]:
!zip -r disaster_response_v2_artifacts.zip \
    models/weights/*.pt \
    config/model_registry.json \
    models/benchmark_report.json

try:
    from google.colab import files
    files.download("disaster_response_v2_artifacts.zip")
    print("[OK] Download initiated for disaster_response_v2_artifacts.zip")
except Exception as e:
    print(f"Artifact zip created at disaster_response_v2_artifacts.zip: {e}")

  adding: models/weights/stage1_mobilenetv3_india_v2.pt (deflated 9%)
  adding: models/weights/stage2_flood_unet_v2.pt (deflated 11%)
  adding: models/weights/stage2_road_passability_v2.pt (deflated 12%)
  adding: models/weights/stage2_structural_rescuenet_v2.pt (deflated 12%)
  adding: config/model_registry.json (deflated 75%)
  adding: models/benchmark_report.json (deflated 69%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[OK] Download initiated for disaster_response_v2_artifacts.zip
